# 验证集对比：base vs GRPO（seed 1）

在 `data/medqa_split_seed42.json` 的 **500 道验证题**上比较原始模型和 GRPO 训练后的模型，判断 GRPO 有没有改变模型（准确率、校准），再决定 seed 2 用不用现在的设置。

两种评估方式（两个模型都是 **bf16 不量化** 加载，和 GRPO 训练时一致）：

| 方式 | 脚本 | 做法 | confidence |
|---|---|---|---|
| 直接作答 `direct` | `eval/evaluate.py --quantization none` | 要求只输出字母，看第一个 token | A–D 的概率 |
| 推理后作答 `cot` | `eval/cot_eval.py` | **和 GRPO 训练相同的 prompt**，贪心生成，最后写 `Answer: X` | `Answer:` 后字母 token 的概率 |
| 采样投票 `cot_vote8` | 同上 | 同样的 prompt 采样 8 次（temperature 1.0，和训练相同） | 8 次里选这个答案的比例 |

`cot_vote8` 的 `sample_acc`（采样回答的平均正确率）可以直接和训练时的 reward（约 0.50）对比。

按顺序运行：检查 GPU → 设置变量 → 拉取 repo → 安装依赖 → **重启会话** → 从"设置变量"重新运行到最后。运行时选 **A100**，总共约 20–30 分钟。

结果写到 Drive 的 `RESULTS_DIR`；已经跑完的评估会自动跳过，断线后从头重跑即可。

In [ ]:
# 1. 检查 GPU 和 Python 版本
!nvidia-smi
import sys, torch
print('Python', sys.version.split()[0])
print('torch', torch.__version__, '| CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| compute {major}.{minor} →', 'bf16' if major >= 8 else 'fp16')
assert torch.cuda.is_available(), '没有 GPU：运行时 → 更改运行时类型 → GPU'
assert sys.version_info >= (3, 11), 'TRL 1.16 需要 Python >= 3.11'

In [ ]:
# 2. 设置变量
REPO_URL = 'https://github.com/skjdsk/medqa-calibration.git'
BRANCH   = 'main'
REPO_DIR = '/content/medqa-calibration'
TRL_VERSION = '1.16.0'
USE_VLLM = True  # 第 4 格按训练时同样的版本装 vLLM

DRIVE_ROOT = '/content/drive/MyDrive/medqa-calibration'
# 要评估的 adapter：{模型名: adapter 目录}；base 不需要 adapter
ADAPTERS = {
    'grpo_seed1': f'{DRIVE_ROOT}/grpo/grpo_train5000_g8_ep1_len1024_seed1_bf16/adapter',
}
RESULTS_DIR = f'{DRIVE_ROOT}/eval_val'
N_SAMPLES   = 8      # cot_vote 的采样次数（和训练时每题 8 个回答一致）
N_DEBUG     = None   # 先想试几题就改成 20；正式对比用 None（全部 500 题）

In [ ]:
# 3. git clone / pull repo
# 私有 repo：在 Colab 左侧 🔑（Secrets）里添加 GITHUB_TOKEN（GitHub 个人访问令牌），并打开"笔记本访问"
import os, subprocess
from pathlib import Path

def sh(cmd, shown=None):
    print('$', shown or cmd)
    subprocess.run(cmd, shell=True, check=True)

clone_url = REPO_URL
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
    if gh_token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://{gh_token}@')
except Exception:
    pass  # 没设置 GITHUB_TOKEN 就按公开 repo 处理

if (Path(REPO_DIR) / '.git').exists():
    sh(f'git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {clone_url} {REPO_DIR}', shown=f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sh('git log -1 --oneline')
for f in ['eval/evaluate.py', 'eval/cot_eval.py', 'eval/metrics.py', 'train/grpo/grpo_data.py', 'train/grpo/rewards.py', 'data/medqa_split_seed42.json']:
    assert Path(f).exists(), f'repo 里找不到 {f}：确认文件已经 push 到 {BRANCH}'
print('cwd =', os.getcwd())

In [ ]:
# 4. 安装依赖
from pathlib import Path
ip = get_ipython()
if Path('requirements.txt').exists():
    ip.run_line_magic('pip', 'install -q -r requirements.txt')
ip.run_line_magic('pip', f'install -q "trl[peft,quantization]=={TRL_VERSION}" "matplotlib" "pandas" "pytest"')
if USE_VLLM:
    # vLLM 会安装它自己需要的 torch 版本，装完后通常要 运行时 → 重启会话，再从"设置变量"那格重新运行
    # vLLM 0.28+ 删掉了 bitsandbytes 量化，而 TRL 在 QLoRA（4-bit）时会让 vLLM 用 bitsandbytes 加载模型
    # → 固定 0.27.1（最后一个支持 bitsandbytes 的版本，且在 TRL 1.16 允许的 0.22–0.31 范围内，要求 torch==2.13.0）
    ip.run_line_magic('pip', f'install -q "trl[vllm]=={TRL_VERSION}" "vllm==0.27.1"')

# Colab 预装的 torchao 0.10 太旧：peft 0.21 在不量化（--quantization none）加 LoRA 时会直接报错
# "only versions above 0.16.0 are supported"。我们用不到 torchao（vllm / peft 都不依赖它），直接卸载
ip.system('pip uninstall -y -q torchao')

import importlib.metadata as md
for pkg in ['torch', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate', 'datasets'] + (['vllm'] if USE_VLLM else []):
    try:
        print(f'{pkg:14s}', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:14s} 未安装')
print('\n如果上面 pip 提示需要重启（或之后 import 报错）：运行时 → 重启会话，然后从"设置变量"那格重新运行。')

**装完依赖后：菜单 运行时 → 重启会话**，然后从第 2 格（设置变量）开始往下运行（跳过第 4 格）。

In [ ]:
# 5. 挂载 Google Drive，检查 adapter 是否存在
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
os.chdir(REPO_DIR)
for name, path in ADAPTERS.items():
    ok = Path(path, 'adapter_config.json').exists()
    print(f'{name:12s}', '✅' if ok else '❌ 找不到', path)
    assert ok, f'{name} 的 adapter 不存在：检查路径，或等 Drive 同步完'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('结果目录：', RESULTS_DIR)

In [ ]:
# 6.（可选）设置 HF_TOKEN：Qwen2.5 和 MedQA 都是公开的，不设也能下载；设置后下载更稳定、不容易被限速
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('已设置 HF_TOKEN')
except Exception:
    print('没有找到 HF_TOKEN（可选），继续匿名下载')

In [ ]:
# 7. 单元测试（几秒钟）
!python -m pytest -q eval/test_cot_eval.py eval/test_metrics.py train/grpo/test_rewards.py

In [ ]:
# 8. 运行评估的小工具：结果文件已经存在就跳过（断线重跑不会重复花时间）
import subprocess
N_ARG = f' --n {N_DEBUG}' if N_DEBUG else ''

def run_eval(cmd, expected_csv):
    if Path(RESULTS_DIR, expected_csv).exists():
        print('已存在，跳过：', expected_csv)
        return
    print('$', cmd)
    get_ipython().system(f'{cmd} 2>&1 | tee -a {RESULTS_DIR}/eval_log.txt')
    assert Path(RESULTS_DIR, expected_csv).exists(), f'没有生成 {expected_csv}：看上面的报错'

MODELS = {'base': None, **ADAPTERS}

In [ ]:
# 9. 直接作答（letter-logit），每个模型约 1–2 分钟
for name, adapter in MODELS.items():
    ad = f' --adapter {adapter}' if adapter else ''
    run_eval(f'python -m eval.evaluate --dataset medqa --medqa_subset val --quantization none'
             f'{ad} --model_name {name} --out_dir {RESULTS_DIR}{N_ARG}',
             f'preds_{name}_medqa4_val_direct.csv')

In [ ]:
# 10. 推理后作答（GRPO 训练 prompt，vLLM），每个模型约 5–10 分钟
#     每个模型单独起一个进程：vLLM 跑完一个模型后显存才能完全释放
for name, adapter in MODELS.items():
    ad = f' --adapter {adapter}' if adapter else ''
    run_eval(f'python -m eval.cot_eval{ad} --model_name {name} --n_samples {N_SAMPLES} '
             f'--out_dir {RESULTS_DIR}{N_ARG}',
             f'preds_{name}_medqa4_val_cot_vote{N_SAMPLES}.csv' if N_SAMPLES else f'preds_{name}_medqa4_val_cot.csv')

In [ ]:
# 11. 汇总：准确率、ECE、Brier、NLL 等（eval/metrics.py），外加配对比较
import pandas as pd
from IPython.display import display, Image

!python -m eval.metrics --pred_csv "{RESULTS_DIR}/preds_*_medqa4_val_*.csv" --out_dir {RESULTS_DIR}/metrics > /dev/null
s = pd.read_csv(f'{RESULTS_DIR}/metrics/summary.csv')
cols = ['eval_mode', 'model_name', 'n', 'accuracy', 'mean_confidence', 'ece', 'brier', 'nll',
        'hc_wrong_rate', 'mean_conf_wrong', 'mean_letter_mass']
display(s[cols].sort_values(['eval_mode', 'model_name']).round(4))

# cot 的额外信息：格式、截断、长度；cot_vote 的采样正确率（≈ 训练 reward）
for mode, extra in [('cot', ['format_ok', 'truncated', 'n_completion_tokens']), (f'cot_vote{N_SAMPLES}', ['sample_acc', 'format_ok'])]:
    parts = [pd.read_csv(f'{RESULTS_DIR}/preds_{m}_medqa4_val_{mode}.csv') for m in MODELS
             if Path(RESULTS_DIR, f'preds_{m}_medqa4_val_{mode}.csv').exists()]
    if parts:
        print(mode); display(pd.concat(parts).groupby('model_name')[extra].mean().round(4))

# 配对比较：同一道题，base 对 / GRPO 错 的题数 vs base 错 / GRPO 对 的题数
# 两个数差不多 → 准确率没有实质变化；|差| 大于约 2×sqrt(两者之和) 才算明显（符号检验的粗略标准）
for mode in ['direct', 'cot', f'cot_vote{N_SAMPLES}']:
    p = Path(RESULTS_DIR, f'preds_base_medqa4_val_{mode}.csv')
    if not p.exists():
        continue
    b = pd.read_csv(p).set_index('question_id')
    for name in ADAPTERS:
        q = Path(RESULTS_DIR, f'preds_{name}_medqa4_val_{mode}.csv')
        if not q.exists():
            continue
        g = pd.read_csv(q).set_index('question_id').loc[b.index]
        lost, gained = int(((b.correct == 1) & (g.correct == 0)).sum()), int(((b.correct == 0) & (g.correct == 1)).sum())
        same = (b.pred == g.pred).mean()
        print(f'{mode:10s} {name}: 答案和 base 相同 {same:.1%} | base 对→{name} 错 {lost} 题 | base 错→{name} 对 {gained} 题'
              f' | 净变化 {gained - lost:+d}（粗略显著线 ±{2 * (lost + gained) ** 0.5:.0f}）')

for f in sorted(Path(RESULTS_DIR, 'metrics').glob('reliability_*.png')):
    print(f.name); display(Image(str(f), width=420))

**怎么读结果**

- **答案和 base 相同的比例很高（比如 > 90%），且准确率、ECE、Brier 都差不多** → GRPO 基本没改变模型。用同样设置跑 seed 2 意义不大，应该先调大学习率（例如 5e-5），seed 1 和 seed 2 都用新设置重跑。
- **cot 或 cot_vote 下 GRPO 的准确率 / 校准明显不同**（净变化超过显著线，或 ECE 差 0.02 以上）→ GRPO 有效果，直接用现在的设置跑 seed 2。
- `direct` 和 `cot` 的结论不一样也很正常：GRPO 只在 CoT 格式上训练过。